In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · Agent identity and principals

**Primer sections:** §3.1 principals · §3.2 own vs delegated authority · §3.3 what agent identity should mean.

The first of the five ideas in the primer: *every agent is a first-class principal*. On Google
Cloud that is **Agent Identity** — a SPIFFE ID per deployed agent resource, expressed to IAM as a
`principal://` member, attested by a runtime-issued X.509 certificate, with access tokens
cryptographically **bound** to that certificate. This notebook builds each of those pieces with
the local twins in `agentsec.identity` and shows the failure modes you should be able to
demonstrate: a `principalSet` that must match exact segments, and a bound token
that is useless when replayed from another certificate.

In [ ]:
from agentsec.logging_utils import quiet_logs

quiet_logs()

import datetime as dt
import json

import jwt  # PyJWT, only used to *peek* at unverified claims for display

from agentsec.identity import (
    AgentIdentity,
    AuthorityContext,
    AuthorityMode,
    BindingMismatch,
    LocalRuntimeCA,
    PrincipalSet,
    TokenIssuer,
    UserPrincipal,
    member_matches,
)


def peek(token: str) -> dict:
    """Display helper: decode WITHOUT verifying. Never do this for authorization decisions."""
    return jwt.decode(token, options={"verify_signature": False})

## 1. One agent, one SPIFFE ID

An Agent Engine deployment gets an identity of the form
`spiffe://agents.global.org-ORG.system.id.goog/resources/aiplatform/projects/N/locations/L/reasoningEngines/ID`.
The *trust domain* is per organisation (or per project when there is no org); the *resource path*
is the deployed resource. IAM spells the same identity `principal://…`.

In [ ]:
agent = AgentIdentity.for_agent_engine(
    project_number="987654321098", location="us-central1", engine_id="support-agent", org_id="123456789012"
)
print("SPIFFE ID     :", agent.spiffe_id)
print("IAM member    :", agent.iam_principal)
print("trust domain  :", agent.trust_domain)
print("service       :", agent.service)
print("project       :", agent.project_number)
print("platformContainer attribute:", agent.platform_container)
print("short name    :", agent.short_name)

# Round-trips: IAM member <-> SPIFFE ID are the same identity.
assert AgentIdentity.parse(agent.iam_principal) == agent == AgentIdentity.parse(agent.spiffe_id)

# Without an organisation the trust domain is per project.
no_org = AgentIdentity.for_agent_engine(project_number=42, location="europe-west1", engine_id="x")
print("no-org trust domain:", no_org.trust_domain)

## 2. Fleet-level bindings: `principalSet://` matches **exact segments**

Two attributes exist: `attribute.platformContainer/aiplatform/projects/N` (every agent in one
project) and `attribute.platform/aiplatform` (every agent on the platform in the org). The match is
on whole path segments — project `987654321098` must **not** be selected by a set written for
`98765432109`. A substring/prefix implementation is the classic bug (drill 8 in notebook 09).

In [ ]:
TD = "agents.global.org-123456789012.system.id.goog"
other_project_agent = AgentIdentity.for_agent_engine(
    project_number="111111111111", location="us-central1", engine_id="marketing-agent", org_id="123456789012"
)

project_set = PrincipalSet.parse(f"principalSet://{TD}/attribute.platformContainer/aiplatform/projects/987654321098")
prefix_set  = PrincipalSet.parse(f"principalSet://{TD}/attribute.platformContainer/aiplatform/projects/98765432109")
platform_set = PrincipalSet.parse(f"principalSet://{TD}/attribute.platform/aiplatform")
other_org_set = PrincipalSet.parse("principalSet://agents.global.org-999.system.id.goog/attribute.platform/aiplatform")

rows = [
    ("project set  → support-agent", project_set.matches(agent)),
    ("project set  → marketing-agent (other project)", project_set.matches(other_project_agent)),
    ("PREFIX set   → support-agent (must be False!)", prefix_set.matches(agent)),
    ("platform set → support-agent", platform_set.matches(agent)),
    ("platform set → marketing-agent", platform_set.matches(other_project_agent)),
    ("other org    → support-agent", other_org_set.matches(agent)),
]
for label, ok in rows:
    print(f"{label:<50} {ok}")

assert project_set.matches(agent) and not project_set.matches(other_project_agent)
assert not prefix_set.matches(agent)

`member_matches` is what the policy engine calls: it understands `principal://`, `spiffe://` and
`principalSet://`. Human and service-account members (`user:`, `serviceAccount:`, `group:`) never
select an agent — an agent identity is a different kind of principal, not a service account with a
long name.

In [ ]:
members = [
    agent.iam_principal,
    agent.spiffe_id,
    f"principalSet://{TD}/attribute.platformContainer/aiplatform/projects/987654321098",
    "user:ana@customer.example",
    "serviceAccount:support@demo-project.iam.gserviceaccount.com",
    f"principal://{TD}/resources/aiplatform/projects/987654321098/locations/us-central1/reasoningEngines/other",
]
for m in members:
    print(f"{member_matches(m, agent)!s:<6} {m}")

## 3. Runtime-attested certificate with the SPIFFE ID in the SAN

On Agent Engine / Cloud Run the runtime provisions a per-agent X.509 certificate (24-hour
validity, auto-rotated) whose Subject Alternative Name is the SPIFFE URI. Nothing like a
service-account key exists for an agent identity; the certificate *is* the credential root.
`LocalRuntimeCA` stands in for the runtime.

In [ ]:
ca = LocalRuntimeCA()
cert = ca.issue(agent)  # default ttl = 24h, fresh key pair, SPIFFE URI SAN

print("SAN URI      :", cert.spiffe_id)
print("subject      :", cert.certificate.subject.rfc4514_string(), "| issuer:", cert.certificate.issuer.rfc4514_string())
print("valid until  :", cert.not_after.isoformat(timespec="minutes"))
print("x5t#S256     :", cert.thumbprint)
print("PEM head     :", cert.pem().splitlines()[0], "...")

assert cert.spiffe_id == agent.spiffe_id
assert ca.verify(cert.certificate) and cert.is_valid_at()
assert cert.not_after - dt.datetime.now(dt.UTC) < dt.timedelta(hours=25)

# A certificate from some other CA (another runtime, an attacker's laptop) does not verify here.
assert not LocalRuntimeCA("rogue-ca").verify(cert.certificate)
print("rogue CA cert accepted by our runtime CA? False")

## 4. Certificate-bound access tokens (RFC 8705, `cnf.x5t#S256`)

The agent's *own-authority* token carries a `cnf` (confirmation) claim with the thumbprint of its
certificate. A resource server that terminates mTLS compares the presented client certificate's
thumbprint with `cnf` — a token that leaks out of the runtime cannot be used without the private
key of that certificate. This is what "certificate-bound" means for Agent Identity (and Agent
Gateway adds DPoP on top: "double-bound").

In [ ]:
issuer = TokenIssuer()  # local STS; on GCP the runtime/metadata server does this for you
token = issuer.mint_agent_token(cert, audience="https://api.acme.example", scope="orders:read")

claims = peek(token)
print(json.dumps({k: claims[k] for k in ("iss", "sub", "aud", "scope", "authority", "cnf")}, indent=2))

verified = issuer.verify(token, audience="https://api.acme.example", presented_thumbprint=cert.thumbprint)
print("verified subject:", verified.subject.rsplit('/', 1)[-1], "| delegated?", verified.is_delegated)
assert verified.cnf["x5t#S256"] == cert.thumbprint

### Replay from another certificate fails

Suppose the token is exfiltrated (a log line, a compromised container). The attacker runs in a
different runtime with a different certificate — or has no certificate at all and presents the
token as a plain bearer. Both fail with `BindingMismatch`, before any audience or scope check
even matters.

In [ ]:
stolen_runtime_cert = ca.issue(agent)  # same agent name, *different* certificate (e.g. attacker's container)

def try_verify(label, **kw):
    try:
        issuer.verify(token, audience="https://api.acme.example", **kw)
        print(f"{label:<45} accepted")
    except BindingMismatch as e:
        print(f"{label:<45} REJECTED  BindingMismatch: {e}")

try_verify("legitimate runtime (matching thumbprint)", presented_thumbprint=cert.thumbprint)
try_verify("replay from another certificate", presented_thumbprint=stolen_runtime_cert.thumbprint)
try_verify("replay as plain bearer (no certificate)")

for bad in ({"presented_thumbprint": stolen_runtime_cert.thumbprint}, {}):
    try:
        issuer.verify(token, audience="https://api.acme.example", **bad)
        raise AssertionError("replay should have failed")
    except BindingMismatch:
        pass

## 5. Own authority vs delegated authority — an explicit object, not a vibe

Every action runs under exactly one authority mode. `AuthorityContext` travels with the request so
policy, credentials and audit all agree. `audit_identities()` is the pair that must appear in every
audit record: the agent, the user (when delegated), and the mode. On Google Cloud, Cloud Audit Logs
show both identities when an agent acts through Auth Manager on a user's behalf.

In [ ]:
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

own = AuthorityContext.own(agent, scopes={"kb:read"})
delegated = AuthorityContext.delegated(agent, ana, scopes={"orders:read", "payments:refund"})

print("own       :", own.audit_identities())
print("delegated :", delegated.audit_identities())
print("chain     :", [c.rsplit('/', 1)[-1] for c in delegated.chain])

# A further hop to a sub-agent keeps the user and the scopes; only the chain grows.
refunds_subagent = AgentIdentity.for_agent_engine(
    project_number="987654321098", location="us-central1", engine_id="refunds-subagent", org_id="123456789012"
)
hop = delegated.with_hop(refunds_subagent)
print("after hop :", hop.audit_identities()["agent"].rsplit('/', 1)[-1], "| chain:", [c.rsplit('/', 1)[-1] for c in hop.chain], "| scopes:", sorted(hop.scopes))
assert hop.user == ana and hop.scopes == delegated.scopes and len(hop.chain) == 2

# Reconstructing authority from a verified token: the agent token above has no `act` → own authority.
from_token = AuthorityContext.from_claims(verified)
assert from_token.mode is AuthorityMode.OWN and from_token.agent == agent and from_token.user is None
print("from token:", from_token.audit_identities())

## On Google Cloud

| Property | Implementation |
|---|---|
| Unique per agent | one SPIFFE ID per `reasoningEngines` / Cloud Run agent service (`--identity-type=agent-identity`) |
| Strongly attested | runtime-issued X.509, 24 h, auto-renewed; ADC returns the cert-bound token |
| No long-lived secrets | no SA-style keys can be created for an agent identity |
| Cannot be impersonated | `generateAccessToken`-style impersonation is not supported |
| Sender-constrained | mTLS-bound tokens; DPoP across Agent Gateway; a default Context-Aware Access policy rejects unbound use |
| Governable as a group | `principalSet://…/attribute.platformContainer/aiplatform/projects/N` in allow, deny, PAB and VPC-SC rules |

Two facts worth knowing: migrating from a service account creates a **new principal with no inherited
permissions** (pre-grant with Policy Analyzer), and
`GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=False` disables binding — a finding, not a setting.

**In one sentence:** "Each agent gets its own SPIFFE identity with a runtime-attested
certificate; its tokens are bound to that certificate so theft doesn't help. Fleet policy uses
principalSets that match exact path segments. And every action carries an explicit authority
mode — own or delegated — so the log shows the agent *and* the user."